# Lineout Deep Dive Chart Experiments

This notebook is a safe sandbox for testing changes to the Python chart functions that power the Lineout Deep Dive page before porting those changes into `python/charts.py`.

## Workflow

1. Build/load the canonical backend DB.
2. Generate baseline legacy panel charts from current `charts.py`.
3. Query source lineout rows into a DataFrame for experimentation.
4. Prototype chart logic in this notebook.
5. Export scratch specs into `data/charts/dev/` and compare visually.
6. Port accepted changes back to `python/charts.py`.

In [20]:
from pathlib import Path
import os
import sys

import altair as alt
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == 'python' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

os.chdir(project_root)

from python.backend import BackendConfig, BackendDatabase
from python.charts import lineout_analysis_panel_chart, lineout_success_by_zone_chart, alt_theme

alt.themes.register('custom_theme', alt_theme)

project_root

PosixPath('/Users/samlindsay/Documents/Projects/Personal/EGRFC/egrfc-stats')

In [2]:
# Prefer primary DB; fall back to alt DB when the primary file is locked.
db_path = 'data/egrfc_backend.duckdb'
try:
    db = BackendDatabase(config=BackendConfig(db_path=db_path))
except RuntimeError as exc:
    if 'locked by another process' in str(exc):
        db_path = 'data/egrfc_backend_alt.duckdb'
        db = BackendDatabase(config=BackendConfig(db_path=db_path))
    else:
        raise

db.build(refresh_pitchero=False, export=False)
db_path

INFO:root:Loaded 715 matches from consolidated file


Removed 8 unfulfilled Pitchero fixtures (0-0 or non-numeric scores)
Removing 1 Pitchero-only rows overlapping with Google games.


'data/egrfc_backend.duckdb'

In [21]:
dev_dir = Path('data/charts/dev')
dev_dir.mkdir(parents=True, exist_ok=True)

# Baseline outputs from current production logic in python/charts.py
baseline = {}
for breakdown in ['numbers', 'area', 'call', 'thrower', 'jumper']:
    for panel in ['breakdown', 'trend']:
        out_file = dev_dir / f'baseline_lineout_{panel}_{breakdown}.json'
        baseline[(breakdown, panel)] = lineout_analysis_panel_chart(
            db,
            breakdown=breakdown,
            panel=panel,
            output_file=str(out_file),
            bind_params=True,
        )

zone_chart = lineout_success_by_zone_chart(
    db,
    output_file=str(dev_dir / 'baseline_lineout_success_by_zone.json')
)

list(sorted(str(p) for p in dev_dir.glob('baseline_lineout*.json')))[:5]

['data/charts/dev/baseline_lineout_breakdown_area.json',
 'data/charts/dev/baseline_lineout_breakdown_call.json',
 'data/charts/dev/baseline_lineout_breakdown_jumper.json',
 'data/charts/dev/baseline_lineout_breakdown_numbers.json',
 'data/charts/dev/baseline_lineout_breakdown_thrower.json']

In [137]:
# Source rows used by lineout panel charts (same shape as charts.py query).
lineouts_df = db.con.execute(
    """
    SELECT
        L.game_id,
        G.date,
        G.squad,
        G.season,
        G.game_type,
        G.opposition,
        L.numbers,
        L.area,
        L.call,
        L.call_type,
        CASE 
            WHEN (L.drive OR L.flyby OR L.crusaders) AND L.won THEN 'Cold' 
            WHEN won THEN 'Hot'
            ELSE 'Lost'
        END AS play,
        J.name AS jumper_name,
        T.name AS thrower_name,
        J.short_name AS jumper,
        T.short_name AS thrower,
        L.won
    FROM lineouts L
    JOIN games G USING (game_id)
    JOIN players J ON L.jumper = J.name
    JOIN players T ON L.thrower = T.name
    WHERE L.won IS NOT NULL
    """
).df()

lineouts_df.head()

,game_id,date,squad,season,game_type,opposition,numbers,area,call,call_type,play,jumper_name,thrower_name,jumper,thrower,won
0,2021-08-28_1st_London_Irish,2021-08-28,1st,2021/22,Friendly,London Irish,5,Middle,,Other,Hot,Sam Lindsay-McCall,Ben Tottman,Sam L,Ben T,True
1,2021-08-28_1st_London_Irish,2021-08-28,1st,2021/22,Friendly,London Irish,5,Middle,RD,Old,Hot,Sam Lindsay-McCall,Ben Tottman,Sam L,Ben T,True
2,2021-08-28_1st_London_Irish,2021-08-28,1st,2021/22,Friendly,London Irish,6,Front,Red,Old,Hot,Gus Nye,Ben Tottman,Gus N,Ben T,True
3,2021-08-28_1st_London_Irish,2021-08-28,1st,2021/22,Friendly,London Irish,4,Front,Snap,4-man only,Hot,Gus Nye,Ben Tottman,Gus N,Ben T,True
4,2021-08-28_1st_London_Irish,2021-08-28,1st,2021/22,Friendly,London Irish,6,Middle,Even,Old,Cold,Sam Lindsay-McCall,Ben Tottman,Sam L,Ben T,True


## Experiment Cell (edit freely)

Use this cell to prototype alternative logic (for example: min-attempt thresholds, alternate sorting, labels, color scales, or tooltip content). Keep the function name temporary and only port stable changes back to `python/charts.py`.

In [145]:
params = {
    "numbers": {
        "scale": alt.Scale(domain=['4', '5', '6', '7'], range=['#7d96e8', '#202946', "#994C4C", "#991515"]),
        "sort": ['4', '5', '6', '7'],
        "step": 60,
        "size": 50
    },
    "area": {
        "scale": alt.Scale(domain=['Front', 'Middle', 'Back'], range=['#991515', 'goldenrod', "#146f14"]),
        "sort": ['Front', 'Middle', 'Back'],
        "step": 90,
        "size": 75,
        "field_title": "Zone"
    },
    "jumper": {
        "scale": alt.Scale(scheme='category20c'),
        "sort": "-y",
        "step": 35,
        "size": 30,
        "label_angle": -30,
        "field_tooltip": "jumper_name",
    },
    "thrower": {
        "scale": alt.Scale(scheme='category20c'),
        "sort": "-y",
        "step": 60,
        "size": 50,
        "label_angle": -30,
        "field_tooltip": "thrower_name",
    },
    "play": {
        "scale": alt.Scale(domain=['Hot', 'Cold', 'Lost'], range=['white', 'blue', '#991515']),
        "stroke": alt.Stroke("play:N", scale=alt.Scale(domain=['Hot', 'Cold', 'Lost'], range=['black', 'transparent', 'transparent'])),
        "sort": ['Hot', 'Cold', 'Lost'],
        "step": 90,
        "size": 75,
    },
    "season": {
        "scale": alt.Scale(scheme='category10'),
        "sort": alt.SortField(field='season', order='ascending'),
        "step": 60,
        "size": 50,
    }
}


numbers_scale = alt.Scale(domain=['4', '5', '6', '7'], range=['#7d96e8', '#202946', "#994C4C", "#991515"])

def lineout_panel_experiment(df: pd.DataFrame, field: str = 'numbers', min_attempts: int = 1):
    work = df.copy()
    work['won'] = work['won'].astype(int)

    group_columns = [field]
    if params.get(field, {}).get('group'):
        group_columns.append(params[field]['group'])
    if params.get(field, {}).get('field_tooltip'):
        group_columns.append(params[field]['field_tooltip'])

    grouped = (
        work.groupby(group_columns, as_index=False)
        .agg(attempts=('won', 'size'), won=('won', 'sum'))
    )
    grouped = grouped[grouped['attempts'] >= min_attempts].copy()
    grouped['success_rate'] = grouped['won'] / grouped['attempts']

    bar = alt.Chart(grouped).mark_bar(
            color='#7d96e8', 
            opacity=0.9, 
            size=params[field]['size'] if field in params else 40
        ).encode(
            x=alt.X(
                f'{field}:N', 
                title=params[field]['field_title'] if field in params and 'field_title' in params[field] else field.title(), 
                sort=params[field]['sort'] if field in params else None, 
                axis=alt.Axis(labelAngle=params[field]['label_angle'] if field in params and 'label_angle' in params[field] else 0)
            ),
            y=alt.Y(
                'attempts:Q', 
                title='Lineouts Taken'
            ),
            stroke=params[field]['stroke'] if field in params and 'stroke' in params[field] else alt.value('transparent'),
            color=alt.Color(
                f'{field}:N', 
                scale=params[field]['scale'], legend=None
            ) if field in params else alt.value('#7d96e8'),
            tooltip=[
                alt.Tooltip(
                    f'{params[field]["field_tooltip"] if field in params and "field_tooltip" in params[field] else field}:N', 
                    title=params[field]['field_title'] if field in params and 'field_title' in params[field] else field.title()),
                alt.Tooltip('attempts:Q', title='Attempts'),
                alt.Tooltip('won:Q', title='Won'),
                alt.Tooltip('success_rate:Q', title='Success Rate', format='.1%'),
            ]
        )

    line = alt.Chart(grouped).mark_line(
            point=alt.OverlayMarkDef(
                filled=False, 
                fill="white", 
                stroke="#202946", 
                size=params[field]['size']*2 if field in params else 40
            ), 
            stroke="#202946", 
            strokeWidth=2.5
        ).encode(
            x=alt.X(
                f'{field}:N', 
                title=field.title(), 
                sort=params[field]['sort'] if field in params else None,
            ),
            y=alt.Y(
                'success_rate:Q', 
                title='Success Rate', 
                axis=alt.Axis(format='%'), 
                scale=alt.Scale(domain=[0, 1])
            ),
        )
    
    text = alt.Chart(grouped).mark_text(
        align='left',
        baseline='middle',
        dx=5,
        dy=-10,
        fontSize=12,
        color='#202946'
    ).encode(
        x=alt.X(
            f'{field}:N', 
            title=field.title(), 
            sort=params[field]['sort'] if field in params else None
        ),
        y=alt.Y(
                'success_rate:Q', 
                title='Success Rate', 
                axis=alt.Axis(format='%'), 
                scale=alt.Scale(domain=[0, 1])
            ),
        text=alt.Text('success_rate:Q', format='.0%')
    )

    line = (line + text).resolve_scale(y='shared')

    chart = (
        alt.layer(bar, line)
        .resolve_scale(y='independent')
        .properties(
            width=alt.Step(params[field]['step'] if field in params else 40), 
            height=300,
            title=alt.Title(
                text=f'{params[field]["field_title"] if field in params and "field_title" in params[field] else field.title()} Breakdown',
            )
        )
    )

    return chart, grouped.sort_values('attempts', ascending=False)

for field in params.keys():
    print(f"=== {field.upper()} ===")
    chart, data = lineout_panel_experiment(lineouts_df, field=field, min_attempts=10)
    chart.show()
    print(data)

=== NUMBERS ===


alt.LayerChart(...)

  numbers  attempts  won  success_rate
3       7       386  283      0.733161
1       5       354  282      0.796610
0       4       320  252      0.787500
2       6       173  132      0.763006
=== AREA ===


alt.LayerChart(...)

     area  attempts  won  success_rate
2  Middle       669  509      0.760837
1   Front       347  285      0.821326
0    Back       216  154      0.712963
=== JUMPER ===


...

       thrower     thrower_name  attempts  won  success_rate
2        Ben T      Ben Tottman       777  613      0.788932
6     George B      George Beet       215  160      0.744186
9       Mark O      Mark Oliver        60   47      0.783333
11    Reggie H    Reggie Harris        56   41      0.732143
7   Harrison B   Harrison Berry        41   29      0.707317
0        Ben D        Ben Darch        29   23      0.793103
8       Josh B  Josh Brimecombe        17   12      0.705882
3        Ben W    Ben Watkinson        11    8      0.727273
=== PLAY ===


alt.LayerChart(...)

   play  attempts  won  success_rate
1   Hot       547  547           1.0
0  Cold       402  402           1.0
2  Lost       284    0           0.0
=== SEASON ===


alt.LayerChart(...)

    season  attempts  won  success_rate
2  2023/24       367  267      0.727520
3  2024/25       318  228      0.716981
1  2022/23       228  185      0.811404
4  2025/26       211  173      0.819905
0  2021/22       109   96      0.880734


In [ ]:
# Save experiment outputs for quick browser inspection.
experiment_spec = dev_dir / 'experiment_lineout_breakdown_numbers.json'
exp_chart.save(str(experiment_spec))

exp_data.head(20)

## Porting Checklist

- Copy only the validated logic into `python/charts.py` (usually `lineout_analysis_panel_chart` and/or `lineout_success_by_zone`).
- Keep the output filenames stable (`lineout_breakdown_*.json`, `lineout_trend_*.json`) so frontend pages continue to work.
- Re-run `python/update.py` to regenerate chart specs.
- Confirm rendering in `lineout-performance.html` and `performance-stats.html`.